## Modelo de Ising

### Breve introducción al Modelo de Ising
El modelo de Ising es un modelo clásico de magnetismo en física estadística que consiste en una red de espines 
$s$ (cada uno tomando valores +1 o -1) interactuando con sus vecinos más próximos. El Hamiltoniano del modelo de Ising es

$$\mathcal{H} = -J\sum_{<i,j>} \sigma_{i} \sigma_{j} -B\sum_{i} \sigma_{i}$$

Donde
  - La suma 

En 2D el modelo muestra una 
transición de fase a temperatura crítica Tc entre un estado ferromagnético (ordenado) y uno paramagnético (desordenado).

En este cuaderno implementaremos una simulación del modelo de Ising en dos dimensiones usando el algoritmo de 
Metropolis:
- Inicializar una red cuadrada de espines aleatorios (o todos alineados).
- Repetir: seleccionar un espín aleatorio, calcular la variación de energía ΔE si se invierte y aceptar la inversión 
  con probabilidad
  $$ min(1, \exp(-ΔE / (k_B T))) .$$
  Aquí tomamos $k_B$ = 1 en unidades adimensionales.
- Medir magnetización y energía después de un número de pasos de equilibrio para obtener promedios en función de la 
  temperatura.

Notas sobre la implementación en este cuaderno:
- Usaremos Julia y la librería `Plots` para visualizar configuraciones y observables.
- Los parámetros importantes son el tamaño de la red `L`, la temperatura `T`, el número de pasos de equilibrado yA el 
  número de muestras para promediar.
- El código siguiente definirá una función `ising()` que inicializa la red, corre la simulación y grafica resultados.

Puedes ejecutar las celdas de código para ajustar parámetros y observar la transición de fase.

In [ ]:
using Base.Threads, Plots, Random, Statistics, LaTeXStrings, DataFrames, CSV, LsqFit
#println("Available threads: ", nthreads())

In [ ]:
# Configurar carpetas para salida (animaciones y gráficos)
const ANIM_DIR = "animations"
const PLOT_DIR = "plots"

# Crear si no existen
mkpath(ANIM_DIR)
mkpath(PLOT_DIR)

In [ ]:
# Implementación refinada del modelo de Ising usando mutable struct
mutable struct IsingModel
    L::Int              # Tamaño de la red (LxL)
    J::Float64          # Fuerza de interacción
    T::Float64          # Temperatura
    spins::Matrix{Int8} # Configuración de espines (±1)
    M::Vector{Float64}  # Magnetización por paso
    E::Vector{Float64}  # Energía por paso
end

In [ ]:
# Función para obtener vecinos (suma de espines adyacentes con condiciones periódicas)
function neighbors(spins::Matrix{Int8}, i::Int, j::Int, L::Int)
    return spins[mod1(i-1, L), j] + spins[mod1(i+1, L), j] + 
           spins[i, mod1(j-1, L)] + spins[i, mod1(j+1, L)]
end

In [ ]:
# Elegir un espín aleatorio y devolver su valor e índices
function random_choose(spins::Matrix{Int8}, L::Int)
    i = rand(1:L)
    j = rand(1:L)
    return spins[i, j], i, j
end

In [ ]:
# Calcular cambio de energía ΔE
function compute_ΔE(sigma_i::Int8, sum_neighbors::Int8, J::Float64)
    return 2 * J * sigma_i * sum_neighbors
end

In [ ]:
function metropolis_ising!(model::IsingModel, steps::Int; rng::AbstractRNG=Random.GLOBAL_RNG, animated::Bool=false, frame_interval::Int=100)
    L, J, T, spins = model.L, model.J, model.T, model.spins
    mag_sum = sum(spins)
    E_current = -J * sum(spins[i,j] * neighbors(spins, i, j, L) for i in 1:L, j in 1:L) / 2 #Obtenido del Hamiltoniano

    M = Vector{Float64}(undef, steps) #Magnetizacion (guardaremos la magnetización firmada por sitio)
    E = Vector{Float64}(undef, steps) #Energia

    frames = animated ? Vector{Matrix{Int8}}() : nothing

    @inbounds for step in 1:steps 
        i = rand(rng, 1:L)
        j = rand(rng, 1:L)
        sigma_i = spins[i, j]
        neigh_sum = neighbors(spins, i, j, L)
        ΔE = compute_ΔE(sigma_i, neigh_sum, J)
        if ΔE ≤ 0 || rand(rng) < exp(-ΔE / T)
            # Actualizar suma de magnetización eficientemente
            mag_sum -= 2 * Int(sigma_i)
            spins[i, j] = -sigma_i
            E_current += ΔE
        end
        # Guardar magnetización firmada por sitio (no el valor absoluto) para poder calcular momentos y Binder
        M[step] = mag_sum / (L * L)
        E[step] = E_current / (L * L)
        if animated && (step % frame_interval == 0)
            push!(frames, copy(spins))
        end
    end

    model.M, model.E, model.spins = M, E, spins
    return model, frames
end

In [ ]:
# Función principal para simular
function ising_model(steps::Int, T::Float64, L::Int, J::Float64=1.0; rng::AbstractRNG=Random.GLOBAL_RNG)
    spins = rand(rng, Int8[-1, 1], L, L) # Configuración inicial aleatoria
    model = IsingModel(L, J, T, spins, Vector{Float64}(), Vector{Float64}())
    T_c = 2 / log(1 + sqrt(2))  # Temperatura crítica para Ising 2D
    
    # Usar Wolff cerca de T_c para acelerar, Metropolis en otros casos
    if abs(T - T_c) < 0.3
        wolff_ising!(model, steps; rng=rng)
    else
        metropolis_ising!(model, steps; rng=rng)
    end
    return model # Retorna el modelo completo con M y E
end

In [ ]:
function compute_magnetization(model::IsingModel)
    eq_start = div(2 * length(model.M), 3) + 1
    mean_mag = mean(abs.(model.M[eq_start:end]))
    return mean_mag
end

function compute_specific_heat(model::IsingModel)
    specific_heat = var(model.E) / (model.T^2)
    return specific_heat
end

function compute_binder_cumulant(model::IsingModel)
    eq_start = div(2 * length(model.M), 3) + 1
    M_eq = model.M[eq_start:end]
    m2 = mean(M_eq .^ 2)
    m4 = mean(M_eq .^ 4)
    U = 1 - (m4 / (3 * m2^2))
    return U
end

function compute_correlations(model::IsingModel)
    eq_start = div(2 * length(model.M), 3) + 1
    final_spins = model.spins # Configuración final de espines
    m_avg = mean(final_spins) # Promedio de espines (magnetización)
    L = model.L
    max_r = div(L, 2)
    correlations = zeros(max_r + 1)
    @inbounds for r in 0:max_r
        corr_sum = 0.0
        for i in 1:L, j in 1:L
            j_shifted = mod1(j + r, L)
            corr_sum += final_spins[i, j] * final_spins[i, j_shifted] - m_avg^2
        end
        correlations[r+1] = corr_sum / (L * L)
    end

    return correlations
end

In [ ]:
function compute_physical_quantities(model::IsingModel)
    magnetization = compute_magnetization(model)
    specific_heat = compute_specific_heat(model)
    correlations = compute_correlations(model)
    binder_cumulant = compute_binder_cumulant(model)
    return magnetization, specific_heat, correlations, binder_cumulant
end

In [ ]:
# Función para simular múltiples temperaturas en paralelo
function simulate_multiple_temperatures(steps::Int, L::Int, temperatures::Vector{Float64}; n_realizations::Int=1) 
    if n_realizations == 1
            results = Vector{IsingModel}(undef, length(temperatures))
            @threads for i in 1:length(temperatures)
                rng = MersenneTwister(threadid() + i)  # RNG único por hilo
                results[i] = ising_model(steps, temperatures[i], L; rng=rng)
            end
    else
        results = Vector{Vector{Float64}}(undef, length(temperatures))
        @threads for i in 1:length(temperatures) 
            accum_scalars = [0.0, 0.0, 0.0]  # [magnetization, specific_heat, binder_cumulant]
            correlations_ensemble = zeros(Float64, div(L, 2) + 1)  # Suponiendo que las correlaciones se almacenan en un vector de tamaño L/2 + 1
            for j in 1:n_realizations
                rng = MersenneTwister(threadid() * 1000 + i * n_realizations + j)  # Seed único por realización
                magnetization, specific_heat, correlations ,binder_cumulant = compute_physical_quantities(ising_model(steps, temperatures[i], L; rng=rng)) # calcular cantidades fisicas de este modelo y guardar
                accum_scalars .+= [magnetization, specific_heat, binder_cumulant]
                correlations_ensemble .+= correlations # Sumar elemento a elemento
            end
            # Calcular promedios            
            results[i] = vcat(accum_scalars, correlations_ensemble) ./ n_realizations
        end
    end
    return results
end

## Animaciones 

### Visualización de las Animaciones Generadas

Las animaciones muestran claramente los diferentes comportamientos del modelo de Ising:

- **T = 1.0 (Fase Ordenada)**: Los espines tienden a alinearse, formando dominios grandes y estables. Se ve mucha coherencia espacial.

- **T ≈ 2.269 (Temperatura Crítica)**: Fluctuaciones intensas con correlaciones a largo alcance. Se forman y destruyen dominios constantemente.

- **T = 3.0 (Fase Desordenada)**: Los espines cambian rápidamente y de forma independiente. No hay estructura coherente.

*Nota: Los archivos GIF se han guardado en el directorio del notebook. Para verlos, abre los archivos `ising_T1.0_comparison.gif`, `ising_T2.269_comparison.gif`, y `ising_T3.0_comparison.gif`.*

In [ ]:
# Animación de la evolución de la configuración de espines
# Para eficiencia: Usa L pequeño (e.g., 20-50), captura frames cada 100 pasos, limita steps a 10^4-10^5

# Función para crear animación
function animate_ising(T::Float64, L::Int=20, steps::Int=10000, frame_interval::Int=100; return_anim::Bool=false, custom_title::String="")
    spins = rand(Int8[-1, 1], L, L)
    model = IsingModel(L, 1.0, T, spins, Vector{Float64}(), Vector{Float64}())
    model, frames = metropolis_ising!(model, steps; animated=true, frame_interval=frame_interval)

    # Crear animación con Plots
    title_text = custom_title != "" ? custom_title : "Ising Model at T=$T"
    anim = @animate for frame in frames
        heatmap(frame, title=title_text, color=:viridis, aspect_ratio=:equal, clims=(-1,1), axis=false, legend=false, ticks=false, colorbar=false, framestyle=:none)
    end

    # Guardar como GIF dentro de ANIM_DIR (fps bajo para eficiencia)
    filename = joinpath(ANIM_DIR, "ising_animation_T$(round(T, digits=3)).gif")
    gif(anim, filename, fps=5)
    println("Animación guardada en: ", filename)

    return return_anim ? anim : model.M
end


In [ ]:
# Función para generar múltiples animaciones)
function animate_multiple_temperatures_display_simple(temperatures::Vector{Float64}, L::Int=20, steps::Int=3000, frame_interval::Int=200)
    # Vector para almacenar las animaciones
    animations = Vector{Any}(undef, length(temperatures))
    
    for i in 1:length(temperatures)
        T = temperatures[i]
        # Crear título descriptivo
        title_text = if T < 2.0
            "T=$T (Ordenado)"
        elseif T < 2.5
            "T=$T (Crítico)"
        else
            "T=$T (Desordenado)"
        end
        # Reutilizar la función animate_ising existente
        anim = animate_ising(T, L, steps, frame_interval; return_anim=true, custom_title=title_text)
        animations[i] = anim
    end
    
    return animations
end

In [ ]:
# Generar y mostrar animaciones contrastantes
contrast_temps = [1.0, 2.269, 4.0]  # Ordenado, Crítico, Desordenado
animations = animate_multiple_temperatures_display_simple(contrast_temps, 40, 20000, 200)

In [ ]:
# Función simple para mostrar animaciones horizontalmente
function show_animations(temperatures::Vector{Float64}; width_px::Int=600)
    html = "<div style='display: flex; gap: 20px; justify-content: center;'>"
    for T in temperatures
        filename = joinpath(ANIM_DIR, "ising_animation_T$(round(T, digits=3)).gif")
        html *= "<img src='$filename' style='width: $(width_px)px; height: auto;' />"
    end
    html *= "</div>"
    display("text/html", html)
end


In [ ]:
# Intentar mostrar las animaciones directamente, si falla mostrar los GIFs
try
    display(plot(animations[1], animations[2], animations[3], layout=(1,3), size=(1600, 900)))
catch
    show_animations(contrast_temps)
end

### Curvas de transición de fase

La magnetización $\langle M \rangle$ es el parámetro de orden del modelo de Ising
$$ M = \frac{1}{N} \langle \sigma_{i} \rangle $$

El calor específico se calcula a partir de las fluctuaciones de la energía (por teorema de fluctuación-disipación)
$$ C= \frac{1}{T^{2}}Var(\mathcal{H}) $$
La función de correlación a una distancia $r$, para una red homogénea y $B=0$ es
$$ G(r) = \langle \sigma_{i} \sigma_{j} \rangle  \approx \frac{1}{M} \sum_{k=1}^{M} (\sigma_{i}\sigma_{i+r})_{k} $$
El cumulante de Binder $U(M)$ se define como
$$ U(M) = 1 -\frac{\langle M ^ 4 \rangle }{3 \langle M ^2 \rangle ^2} $$
donde $M$ es la magnetización del sistema.

In [ ]:
# Ejemplo de uso
temperatures = [2.0, 2.5]
results = simulate_multiple_temperatures(10000, 16, temperatures; n_realizations=10) # Simular 10 realizaciones por temperatura
#means = compute_mean_ensembles(results, temperatures, 5) # Calcular promedios, max 5 espines de distancia para correlaciones
#println(means)  # Debería dar promedios razonables

In [ ]:
function ensemble_multiple_sizes(L_size::Vector{Int}, temperatures::Vector{Float64}, n_realizations::Int, steps::Int)
    # Cambiar Correlations a Vector{Float64} para almacenar el vector completo
    all_data = DataFrame(L=Int[], T=Float64[], Magnetization=Float64[], SpecificHeat=Float64[], BinderCumulant=Float64[], Correlations=Vector{Float64}[])
    
    for L in L_size
        # Llamar a simulate_multiple_temperatures para este L (ya calcula promedios)
        results = simulate_multiple_temperatures(steps, L, temperatures; n_realizations=n_realizations)
        
        for i in 1:length(temperatures)
            # Extraer valores de results[i]: [magnetization, specific_heat, binder_cumulant, correlations[1], correlations[2], ...]
            magnetization = results[i][1]
            specific_heat = results[i][2]
            binder_cumulant = results[i][3]
            correlations = results[i][4:end]  # Vector completo de correlaciones
            
            push!(all_data, (L, temperatures[i], magnetization, specific_heat, binder_cumulant, correlations))
        end
    end
    return all_data
end

In [ ]:
@time dataframe_physical_quantities = ensemble_multiple_sizes([16, 24, 32], collect(1.5:0.1:3.5), 300, 2500000) 

In [ ]:
function plot_by_group(df::DataFrame, x::Symbol, y::Symbol; group::Symbol=:L,
                       xlabel::AbstractString="", ylabel::AbstractString="",
                       size::Tuple{Int,Int}=(900,600), legend=:topright,
                       grid::Bool=false, fmt::Symbol=:pdf, savepath::Union{String,Nothing}=nothing,
                       posthook=nothing) 
    groups = groupby(df, group)
    colors = palette(:Set1, length(groups))  # Cambié :auto por :Set1 para colores discretos y distintivos
    markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :star5, :hexagon, :pentagon]  # Lista de markers variados
    p = plot(xlabel=xlabel, ylabel=ylabel, legend=legend, grid=grid, size=size)

    for (i, g) in enumerate(groups)
        gs = sort(g, x)
        xs, ys = gs[!, x], gs[!, y]
        label = "$group=$(unique(gs[!, group])[1])"
        marker_type = markers[mod1(i, length(markers))]  # Asigna marker cíclicamente
        scatter!(p, xs, ys; marker=marker_type, ms=5, color=colors[i], label=label, markerstrokewidth=0) 
    end

    # Hook opcional (para líneas o ajustes teóricos)
    if posthook !== nothing
        posthook(p)
    end

    if savepath !== nothing
        filename = savepath * "." * string(fmt)
        savefig(p, filename)
        println("Saved: ", filename)
    end

    return p
end

In [ ]:
function plot_physical_quantities_df(df::DataFrame; savepath=nothing)
    p1 = plot_by_group(df, :T, :Magnetization;
                       xlabel=L"$T$",
                       ylabel=L"$\langle M \rangle$")
    p2 = plot_by_group(df, :T, :SpecificHeat;
                       xlabel=L"$T$",
                       ylabel=L"$C_v$")
    combined = plot(p1, p2, layout=(2,1))

    if savepath !== nothing
        savefig(combined, savepath)
    end
    return combined
end

In [ ]:
function plot_binder_cumulant_df(df::DataFrame; savepath=nothing, fmt::Symbol=:pdf)
    Tc_exact = 2 / log(1 + sqrt(2))
    posthook = p -> vline!(p, [Tc_exact],
                           label=L"$T_c$ exacta",
                           linestyle=:dash, color=:black, lw=2)

    p = plot_by_group(df, :T, :BinderCumulant;
                      xlabel=L"$T$",
                      ylabel=L"Binder Cumulant $U$",
                      posthook=posthook, savepath=savepath, fmt=fmt)
    return p
end

In [ ]:
# Función auxiliar para ajustar modelos de correlaciones usando LsqFit
function fit_correlation_model(r_data::Vector{Int}, corr_data::Vector{Float64}, model_func::Function, initial_guess::Vector{Float64})
    fit = curve_fit(model_func, r_data, corr_data, initial_guess)
    return fit.param  # Retorna los parámetros ajustados
end

In [ ]:
function plot_correlations_df(df::DataFrame; L_list=nothing, T_list=nothing,
                              max_r=10, savepath=nothing, fmt::Symbol=:pdf)
    # lógica de selección L/T idéntica
    if L_list === nothing
        L_list = unique(df.L)
    end
    if T_list === nothing
        Ts_all = sort(unique(df.T))
        T_list = LinRange(first(Ts_all), last(Ts_all), 4)  # Cambiado a LinRange para compatibilidad
    end

    # Crear subplots: uno por cada L
    n_L = length(L_list)
    p = plot(layout=(n_L, 1), size=(800, 600), legend=:topright)

    for (idx, L) in enumerate(L_list)
        subplot_idx = idx
        colors = palette(:Set1, length(T_list))  # Cambié :auto por :Set1
        markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :star5, :hexagon, :pentagon]  # Lista de markers
        ci = 1

        for T in T_list
            row = filter(r -> r.L == L && isapprox(r.T, T; atol=1e-8), df)
            if nrow(row) == 0
                @warn "No data for L=$L, T=$T"
                continue
            end
            correlations = row.Correlations[1]
            r_values = 0:(length(correlations)-1)
            label = "T=$(round(T, digits=3))"
            marker_type = markers[mod1(ci, length(markers))]  # Asigna marker por T
            scatter!(p[subplot_idx], r_values, correlations; marker=marker_type, ms=4, color=colors[ci], label=label, markerstrokewidth=0)  # Cambié plot! por scatter!
            ci += 1
        end

        # Agregar curvas ajustadas al subplot actual (inline para compatibilidad con Subplot)
        Tc = 2 / log(1 + sqrt(2))  # Tc exacta
        r_fit = 1:max_r  # Evitar r=0 para ajustes
        selected_temps = [2.2, 3.0]  # Ajusta según tus datos
        
        for T in selected_temps
            row = filter(r -> r.L == L && isapprox(r.T, T; atol=1e-8), df)
            if nrow(row) == 0
                continue
            end
            correlations = row.Correlations[1]
            r_data = Vector{Int}(1:(length(correlations)-1))
            corr_data = correlations[2:end]
            
            if T ≈ 3.0
                model = (x, p) -> p[1] .* exp.(-x ./ p[2])
                initial_guess = [0.5, 1.0]
                params = fit_correlation_model(r_data, corr_data, model, initial_guess)
                G_fitted = model(r_fit, params)
                label_fit = "G(r) ~ $(round(params[1], digits=2)) exp(-r / $(round(params[2], digits=2))) (Disordered, fitted)"
                plot!(p[subplot_idx], r_fit, G_fitted, label=label_fit, linestyle=:dash, color=:purple, linewidth=2.0)
            elseif T ≈ 2.2
                model = (x, p) -> p[1] ./ (x .^ (1 + p[2]))
                initial_guess = [0.2, 0.25]
                params = fit_correlation_model(r_data, corr_data, model, initial_guess)
                G_fitted = model(r_fit, params)
                label_fit = "G(r) ~ $(round(params[1], digits=2)) / r^(1+$(round(params[2], digits=2))) (Critical, fitted)"
                plot!(p[subplot_idx], r_fit, G_fitted, label=label_fit, linestyle=:dash, color=:orange, linewidth=2.0)
            end
        end

        # Agregar título al subplot
        plot!(p[subplot_idx], title=L"L=%$L", xlabel=L"Distance $r$", ylabel=L"$G(r)$")
    end

    if savepath !== nothing
        filename = savepath * "." * string(fmt)
        savefig(p, filename)
        println("Saved: ", filename)
    end
    return p
end

In [ ]:
plot_physical_quantities_df(dataframe_physical_quantities; savepath=joinpath(PLOT_DIR, "physical_quantities.svg"))

In [ ]:
# Gráfico de cumulante de Binder:
plot_binder_cumulant_df(dataframe_physical_quantities, savepath=joinpath(PLOT_DIR,"binder_cumulant"), fmt=:svg)

In [ ]:
# Correlaciones (elige L y T si quieres):
plot_correlations_df(dataframe_physical_quantities, L_list=[24,32], T_list=[2.2, 3.0], savepath=joinpath(PLOT_DIR,"correlations"), fmt=:pdf)

In [ ]:
## Explorando la zona crítica con el algoritmo de Wolff

In [ ]:
function wolff_step!(spins::Matrix{Int8}, L::Int, J::Float64, T::Float64; rng::AbstractRNG=Random.GLOBAL_RNG)
    # Elige semilla aleatoria
    i0, j0 = rand(rng, 1:L), rand(rng, 1:L)
    s0 = spins[i0, j0]
    p_add = 1 - exp(-2 * J / T)       # probabilidad de añadir vecino al cluster

    in_cluster = falses(L, L)
    stack = [(i0, j0)]
    in_cluster[i0, j0] = true

    while !isempty(stack)
        (i, j) = pop!(stack)   # DFS style stack
        # vecinos con condiciones periódicas
        neighs = ((mod1(i-1, L), j), (mod1(i+1, L), j), (i, mod1(j-1, L)), (i, mod1(j+1, L)))
        for (ni, nj) in neighs
            if !in_cluster[ni, nj] && spins[ni, nj] == s0
                if rand(rng) < p_add
                    in_cluster[ni, nj] = true
                    push!(stack, (ni, nj))
                end
            end
        end
    end

    # Calcular ΔE del cluster sumando solo fronteras (vecinos fuera del cluster)
    deltaE = 0.0
    @inbounds for i in 1:L, j in 1:L
        if in_cluster[i, j]
            # Contar vecinos FUERA del cluster
            sum_out = Int8(0)
            neighs = ((mod1(i-1, L), j), (mod1(i+1, L), j), (i, mod1(j-1, L)), (i, mod1(j+1, L)))
            for (ni, nj) in neighs
                if !in_cluster[ni, nj]
                    sum_out += spins[ni, nj]
                end
            end
            # Reciclar compute_ΔE para calcular contribución
            if sum_out != 0
                deltaE += compute_ΔE(spins[i, j], sum_out, J)
            end
        end
    end

    # Voltear el cluster y actualizar magnetización
    cluster_size = 0
    @inbounds for i in 1:L, j in 1:L
        if in_cluster[i, j]
            spins[i, j] = -spins[i, j]
            cluster_size += 1
        end
    end

    return cluster_size, deltaE  # devuelve tamaño y cambio de energía
end

In [ ]:
function wolff_ising!(model::IsingModel, steps::Int; rng::AbstractRNG=Random.GLOBAL_RNG, animated::Bool=false, frame_interval::Int=100)
    L, J, T, spins = model.L, model.J, model.T, model.spins
    mag_sum = sum(spins)
    # Energía inicial (usar neighbors para reciclar código)
    E_current = -J * sum(spins[i,j] * neighbors(spins, i, j, L) for i in 1:L, j in 1:L) / 2 

    M = Vector{Float64}(undef, steps)
    E = Vector{Float64}(undef, steps)
    frames = animated ? Vector{Matrix{Int8}}() : nothing

    for step in 1:steps
        cluster_size, deltaE = wolff_step!(spins, L, J, T; rng=rng)
        # Actualizar magnetización incrementalmente
        mag_sum = sum(spins)  # Mantener simple; optimizable rastreando signo del cluster
        E_current += deltaE   # Actualización incremental de energía (evita recálculo O(L²))

        M[step] = mag_sum / (L * L)
        E[step] = E_current / (L * L)

        if animated && (step % frame_interval == 0)
            push!(frames, copy(spins))
        end
    end

    model.M, model.E, model.spins = M, E, spins
    return model, frames
end
# ...existing code...

In [ ]:
#Implementación del algoritmo de Wolff (próximamente)
 
function ising_model(steps::Int, T::Float64, L::Int, J::Float64; rng::AbstractRNG=Random.GLOBAL_RNG)
    spins = rand(rng, Int8[-1,1], L, L) #Configuración inicial aleatoria
    model = IsingModel(L,J, T, spins, Vector{Float64}(), Vector{Float64}())
    T_c = 2 / log(1 + sqrt(2))  # Temperatura crítica aproximada para el modelo 2D


    if abs(T - T_c) < 0.3   # Umbral para usar Wolff cerca de T_c
        wolff_ising!(model, steps; rng=rng)
    else
        metropolis_ising!(model, steps; rng=rng)
    end
    return model
end
# Nota: El código anterior redefine la función ising_model para usar el algoritmo de Wolff cerca de T_c.

In [ ]:
ising_model(10000, 2.269 , 16; rng=MersenneTwister(42))  # Ejemplo de uso del nuevo ising_model

In [ ]:
@time dataframe_physical_quantities = ensemble_multiple_sizes([16, 24, 32], collect(1.5:0.1:3.5), 300, 2500000) 